# K-Means Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 1

In this exercise we apply K-Means clustering to well log data to propose groupings in
subsurface measurements. In petrophysics, clusters of log responses are used as *candidate*
rock types (electrofacies), fluid zones or completion intervals where no labels exist.
A cluster label is a hypothesis: it becomes a facies only when external evidence
(core, cuttings, depth trends, other wells) supports that name.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

We load `w5.csv`, which contains well log measurements for a single well:
- **DEPTH** — measured depth
- **Pressure** — formation pressure
- **AI** — acoustic impedance
- **VpVs** — compressional-to-shear velocity ratio
- **SW** — water saturation
- **phid** — density porosity
- **vcl** — volume of clay

In [ ]:
df = pd.read_csv("w5.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()
print(f"\nMissing values:\n{df.isnull().sum()}")

---
## 2 — Visualize Feature Distributions

Before clustering it is essential to understand the distribution of each feature.
We plot histograms and a pairplot to look for natural groupings and correlations.

In [ ]:
# Histogram of each feature
features = ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.ravel(), features):
    ax.hist(df[feat].dropna(), bins=40, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(feat)
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
fig.suptitle("Feature Distributions — Well W5", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of the six features (all 7,071 samples; takes a few seconds)
plot_df = df[features].dropna()
sns.pairplot(plot_df, diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15})
plt.suptitle("Pairplot of Well Log Features", y=1.02, fontsize=14)
plt.show()

---
## 3 — Feature Selection and Standardization

K-Means is sensitive to feature scales because it uses Euclidean distance. We select
the petrophysical features (excluding DEPTH) and apply **StandardScaler** to give each
feature zero mean and unit variance.

> **Exploratory analysis of one fixed dataset.** The scaler is fitted on all samples of W5
> because we only describe this well: nothing is predicted for new data, so there is no
> training/validation/test split. If the clusters were used to label other wells, the scaler
> and K-Means would be fitted together (one pipeline) on the training wells only and applied
> to the new wells with `.predict`.

In [ ]:
# Select features for clustering (exclude DEPTH)
feature_cols = ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]
X = df[feature_cols].dropna().copy()
print(f"Samples after dropping NaNs: {X.shape[0]}")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verify: means ~ 0, stds ~ 1
print(f"Scaled means: {X_scaled.mean(axis=0).round(4)}")
print(f"Scaled stds:  {X_scaled.std(axis=0).round(4)}")

---
## 4 — Apply K-Means with K=3

We start with K=3 clusters as an initial guess, to be checked with the diagnostics in
Sections 6–8. A geologist might expect sand, shale and transition zones, but that is a
hypothesis to test in Sections 9–10, not something the algorithm knows.

In [ ]:
kmeans_3 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels_3 = kmeans_3.fit_predict(X_scaled)

print(f"Cluster labels: {np.unique(labels_3)}")
print(f"Cluster sizes:")
for c in np.unique(labels_3):
    print(f"  Cluster {c}: {np.sum(labels_3 == c)} samples")
print(f"\nInertia: {kmeans_3.inertia_:.2f}")

---
## 5 — Visualize Clusters in 2D

We project clusters onto pairs of features to inspect spatial coherence. AI vs VpVs
is a classic crossplot in rock physics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_pairs = [("AI", "VpVs"), ("phid", "SW"), ("AI", "vcl")]
colors = ["steelblue", "tomato", "forestgreen"]

for ax, (xf, yf) in zip(axes, plot_pairs):
    for c in np.unique(labels_3):
        mask = labels_3 == c
        ax.scatter(
            X[xf].values[mask], X[yf].values[mask],
            c=colors[c], label=f"Cluster {c}", alpha=0.5, s=20
        )
    ax.set_xlabel(xf)
    ax.set_ylabel(yf)
    ax.legend()
    ax.set_title(f"{xf} vs {yf}")

fig.suptitle("K-Means (K=3) — Cluster Visualization", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 6 — Elbow Method

The elbow method plots total within-cluster sum of squares (inertia) against the
number of clusters. The "elbow" point — where adding more clusters yields diminishing
returns — suggests candidate values of K. Inertia always falls as K grows, so it cannot
choose K on its own.

In [ ]:
K_range = range(1, 11)
inertias = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_range, inertias, "o-", color="steelblue", linewidth=2, markersize=8)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Elbow Method — Inertia vs K")
ax.set_xticks(list(K_range))
plt.tight_layout()
plt.show()

for k, inertia in zip(K_range, inertias):
    print(f"  K={k:>2d}  Inertia={inertia:>10.2f}")

---
## 7 — Silhouette Analysis

The silhouette score measures how similar each point is to its own cluster compared
to neighbouring clusters. Values near +1 mean well inside the cluster, near 0 on a boundary;
negative values mean closer to another cluster on average. We evaluate K=2 through K=6.

In [ ]:
sil_scores = []
K_sil_range = range(2, 7)

for k in K_sil_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"  K={k}  Silhouette Score={score:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_sil_range), sil_scores, "s-", color="tomato", linewidth=2, markersize=8)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Analysis — Score vs K")
ax.set_xticks(list(K_sil_range))
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plot for each K
fig, axes = plt.subplots(1, len(list(K_sil_range)), figsize=(20, 5))

for ax, k in zip(axes, K_sil_range):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_vals = silhouette_samples(X_scaled, labels)
    avg_score = silhouette_score(X_scaled, labels)

    y_lower = 10
    for c in range(k):
        cluster_sil = np.sort(sil_vals[labels == c])
        y_upper = y_lower + len(cluster_sil)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7)
        y_lower = y_upper + 10

    ax.axvline(avg_score, color="red", linestyle="--", linewidth=1.5)
    ax.set_title(f"K={k} (avg={avg_score:.3f})")
    ax.set_xlabel("Silhouette Value")
    ax.set_yticks([])

axes[0].set_ylabel("Cluster")
fig.suptitle("Silhouette Plots by K", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8 — Choose K and Refit

We choose K with a stated rule: the highest mean silhouette for K = 2–6, checked against
the elbow plot. Mean silhouettes of about 0.3–0.4 indicate weakly separated groups, so the
chosen K is a working choice for this feature set, not evidence that the well contains that
many facies.

In [ ]:
# Chosen K: highest mean silhouette for K = 2-6 (check it against the elbow plot)
optimal_k = list(K_sil_range)[np.argmax(sil_scores)]
print(f"Chosen K (highest mean silhouette, K = 2-6): {optimal_k}")

# Refit with the chosen K
kmeans_opt = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
labels_opt = kmeans_opt.fit_predict(X_scaled)

print(f"\nFinal cluster sizes:")
for c in np.unique(labels_opt):
    print(f"  Cluster {c}: {np.sum(labels_opt == c)} samples")
print(f"\nInertia: {kmeans_opt.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(X_scaled, labels_opt):.4f}")

In [ ]:
# Visualize the clusters for the chosen K
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_pairs = [("AI", "VpVs"), ("phid", "SW"), ("AI", "vcl")]
cmap = plt.colormaps["tab10"].resampled(optimal_k)

for ax, (xf, yf) in zip(axes, plot_pairs):
    scatter = ax.scatter(
        X[xf].values, X[yf].values,
        c=labels_opt, cmap=cmap, alpha=0.5, s=20
    )
    ax.set_xlabel(xf)
    ax.set_ylabel(yf)
    ax.set_title(f"{xf} vs {yf}")

plt.colorbar(scatter, ax=axes[-1], label="Cluster")
fig.suptitle(f"K-Means (K={optimal_k}) — Chosen K", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9 — Interpret Clusters

We compute the mean feature values per cluster in original units and visualize them with a
heatmap in standardized units (for comparison across features). Read physical values from
the original-unit table, and write a hypothesis for what each cluster might represent:
a rock type, a fluid effect or a depth trend.

In [ ]:
# Compute cluster means (original scale)
X_with_labels = X.copy()
X_with_labels["Cluster"] = labels_opt

cluster_means = X_with_labels.groupby("Cluster")[feature_cols].mean()
print("Cluster Means (Original Scale):")
print(cluster_means.round(4).to_string())

In [ ]:
# Heatmap of cluster means (standardized for visual comparison)
cluster_means_scaled = pd.DataFrame(
    scaler.transform(cluster_means),
    columns=feature_cols,
    index=[f"Cluster {c}" for c in cluster_means.index]
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    cluster_means_scaled,
    annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    linewidths=0.5, ax=ax
)
ax.set_title("Cluster Centroids — Standardized Feature Values", fontsize=13)
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots per cluster for each feature
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, feat in zip(axes.ravel(), feature_cols):
    data_to_plot = [X_with_labels.loc[X_with_labels["Cluster"] == c, feat].values
                    for c in range(optimal_k)]
    bp = ax.boxplot(data_to_plot, tick_labels=[f"C{c}" for c in range(optimal_k)], patch_artist=True)
    for patch, color in zip(bp["boxes"], plt.cm.tab10.colors[:optimal_k]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat)
    ax.set_xlabel("Cluster")

fig.suptitle("Feature Distributions by Cluster", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10 — Clusters Along the Well Depth

Plotting cluster assignments against depth shows whether the clusters recur down the well,
as lithologies would, or mainly split it into depth intervals. DEPTH is not a feature, but
logs that trend with depth (for example Pressure, or compaction-sensitive AI and VpVs) can
carry depth into the clustering. Use the plot and the printed checks to write one hypothesis
per cluster and the evidence that would test it (core, cuttings, a rerun without Pressure).

In [ ]:
# Merge cluster labels back with depth
depth_vals = df.loc[X.index, "DEPTH"].values

fig, ax = plt.subplots(figsize=(4, 12))
scatter = ax.scatter(
    labels_opt, depth_vals,
    c=labels_opt, cmap=plt.colormaps["tab10"].resampled(optimal_k),
    s=8, alpha=0.7
)
ax.set_xlabel("Cluster")
ax.set_ylabel("Depth")
ax.invert_yaxis()
ax.set_title(f"Cluster Assignments vs Depth (K={optimal_k})")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

# Evidence for the hypotheses: which logs follow depth, and which depth
# interval does each cluster occupy?
print("Correlation of each feature with DEPTH:")
print(X.corrwith(df.loc[X.index, "DEPTH"]).round(2).to_string())
print("\nDepth by cluster:")
depth_by_cluster = pd.Series(depth_vals, name="DEPTH").groupby(labels_opt).describe()
print(depth_by_cluster[["count", "min", "50%", "max"]].rename_axis("Cluster").round(1))

---
## Summary

In this exercise we covered:

1. **Data Loading and Exploration** — understanding well log feature distributions and correlations.
2. **Feature Standardization** — using StandardScaler before clustering (exploratory: fitted on this one well).
3. **K-Means with K=3** — applying clustering and visualizing results in 2D crossplots.
4. **Elbow Method** — plotting inertia vs K to identify diminishing returns.
5. **Silhouette Analysis** — quantifying cluster separation for K=2 through K=6.
6. **Choosing K** — a stated rule (highest mean silhouette, checked against the elbow); values of 0.3–0.4 mean weakly separated groups.
7. **Cluster Interpretation** — profiling clusters in original units, with heatmaps and box plots.
8. **Depth Integration** — mapping cluster assignments along the well to see whether they follow lithology, fluid or depth.

K-Means proposes candidate groupings of log responses. Treat each cluster label as a
hypothesis: check whether it follows lithology, fluid (SW) or depth (Pressure and compaction
trends), and confirm any electrofacies interpretation with core, cuttings or other wells.